[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/pypath/blob/main/notebooks/module8/07-performance.ipynb)

# Module 8 Lesson 7 — Performance Profiling & Optimization

**Module 8: Best Practices & Real-World Python** | Estimated time: 25 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Use `%timeit` and `%%timeit` magic commands to benchmark code
- Profile CPU usage with `cProfile` and `pstats`
- Use `line_profiler` to find the exact slow line in a function
- Use `memory_profiler` to track memory consumption line by line
- Apply NumPy vectorization to replace slow Python loops (10x-100x speedup)
- Use `np.einsum` for efficient batched operations
- Cache expensive function results with `functools.lru_cache` and `functools.cache`
- Use `functools.cached_property` for lazy class attributes
- Understand Redis caching patterns and `joblib.Memory` for disk caching

In [ ]:
# Install profiling tools
!pip install line_profiler memory_profiler joblib --quiet
print("Profiling tools installed.")

import numpy as np
import time
import functools
import cProfile
import pstats
import io

## %timeit and %%timeit — Quick Benchmarking

`%timeit` runs a single expression many times and reports the best result. `%%timeit` times an entire cell.

In [ ]:
# %timeit — single line
# Python list comprehension
%timeit [i**2 for i in range(10_000)]

# Built-in map
%timeit list(map(lambda i: i**2, range(10_000)))

In [ ]:
%%timeit
# %%timeit — entire cell
total = 0
for i in range(10_000):
    total += i * i

In [ ]:
%%timeit
# NumPy equivalent — how much faster?
import numpy as np
arr = np.arange(10_000)
result = (arr ** 2).sum()

## cProfile — Finding the Bottleneck Function

`cProfile` is Python's built-in deterministic profiler. It counts how many times each function was called and how long it took.

In [ ]:
def slow_function_to_profile():
    """A deliberately slow function with a hidden bottleneck."""
    # Step 1: build a list of primes (cheap)
    primes = [n for n in range(2, 500) if all(n % i != 0 for i in range(2, n))]

    # Step 2: compute Fibonacci (the real bottleneck — recursive!)
    def fib(n):
        if n <= 1:
            return n
        return fib(n - 1) + fib(n - 2)

    fib_numbers = [fib(n) for n in range(25)]  # fib(24) alone makes ~150k calls

    # Step 3: string operations (cheap)
    result = "".join(str(x) for x in fib_numbers[:10])

    return primes, fib_numbers, result


# Profile with cProfile
print("=== cProfile output ===")
with cProfile.Profile() as profiler:
    slow_function_to_profile()

stats = pstats.Stats(profiler, stream=io.StringIO())
stats.sort_stats("cumulative")
stats.print_stats(15)  # show top 15 lines

buf = io.StringIO()
s = pstats.Stats(profiler, stream=buf)
s.sort_stats("cumulative")
s.print_stats(15)
print(buf.getvalue())

In [ ]:
# Profile from the command line (run this in a terminal):
profile_script = """
import cProfile
import pstats

cProfile.run('slow_function_to_profile()', 'profile_output.prof')

# Analyse the saved profile
stats = pstats.Stats('profile_output.prof')
stats.sort_stats('cumulative')   # sort by total time including sub-calls
stats.sort_stats('tottime')      # sort by time in function only
stats.print_stats(20)            # show top 20
stats.print_callers('fib')       # who called fib?
"""
print("cProfile command-line usage:")
print(profile_script)

# Also: python -m cProfile -s cumulative myscript.py
print("Command line: python -m cProfile -s cumulative -o output.prof myscript.py")

## line_profiler — Line-by-Line Profiling

Once cProfile tells you *which function* is slow, `line_profiler` tells you *which line* inside it.

In [ ]:
# Load the line_profiler extension
%load_ext line_profiler

In [ ]:
def process_data(n: int = 50_000) -> float:
    """A function with a mix of fast and slow operations."""
    # Line A: list comprehension (fast)
    data = [x * 0.5 for x in range(n)]

    # Line B: Python sum loop (slow)
    total = 0.0
    for value in data:          # <-- this loop is the bottleneck
        total += value ** 0.5

    # Line C: string formatting (moderate)
    report = ", ".join(f"{v:.2f}" for v in data[:5])

    return total


# Profile line by line using %lprun magic
%lprun -f process_data process_data(50_000)

## memory_profiler — Tracking Memory Usage

In [ ]:
%load_ext memory_profiler

In [ ]:
%%writefile memory_demo.py
from memory_profiler import profile


@profile
def memory_hungry(n: int = 100_000) -> None:
    # Step 1: allocate a large list
    big_list = list(range(n))          # ~800 KB for 100k ints

    # Step 2: create a copy
    copy = big_list[:]                 # another ~800 KB

    # Step 3: delete original
    del big_list                       # should be freed

    # Step 4: convert to set (much more memory)
    as_set = set(copy)                 # sets have more overhead than lists

    # Step 5: process and return
    result = sum(as_set)
    return result


if __name__ == "__main__":
    memory_hungry()

In [ ]:
!python memory_demo.py

## NumPy Vectorization — Replace Loops with Array Operations

NumPy operations execute in C, bypassing Python's interpreter overhead. This is typically **10x to 100x faster** than Python loops.

In [ ]:
import numpy as np

N = 1_000_000
data = list(range(N))
data_array = np.arange(N, dtype=np.float64)

print("=== Benchmark: sum of square roots ===")

# Method 1: Pure Python loop
start = time.perf_counter()
total_py = 0.0
for x in data:
    total_py += x ** 0.5
time_py = time.perf_counter() - start
print(f"Python loop:          {time_py:.4f}s  result={total_py:.2f}")

# Method 2: List comprehension + sum
start = time.perf_counter()
total_lc = sum(x ** 0.5 for x in data)
time_lc = time.perf_counter() - start
print(f"Generator + sum:      {time_lc:.4f}s  result={total_lc:.2f}")

# Method 3: NumPy vectorized
start = time.perf_counter()
total_np = np.sqrt(data_array).sum()
time_np = time.perf_counter() - start
print(f"NumPy vectorized:     {time_np:.4f}s  result={total_np:.2f}")

print(f"\nSpeedup (loop → numpy):  {time_py/time_np:.1f}x faster")

In [ ]:
# More vectorization examples
print("=== Vectorized operations vs loops ===")

A = np.random.rand(1000, 1000)
B = np.random.rand(1000, 1000)

# Element-wise multiplication
%timeit result = A * B                          # vectorized
%timeit result = [[A[i,j]*B[i,j] for j in range(1000)] for i in range(1000)]  # loop (slow!)

In [ ]:
# np.einsum — Einstein summation for batched operations
print("=== np.einsum ===")

# Batch matrix multiplication: (batch, m, k) x (batch, k, n) -> (batch, m, n)
batch = 32
m, k, n = 64, 128, 64
X = np.random.rand(batch, m, k)
W = np.random.rand(batch, k, n)

# Method 1: loop over batch
start = time.perf_counter()
result_loop = np.stack([X[i] @ W[i] for i in range(batch)])
time_loop = time.perf_counter() - start

# Method 2: np.einsum (no loop)
start = time.perf_counter()
result_einsum = np.einsum("bik,bkj->bij", X, W)
time_einsum = time.perf_counter() - start

# Method 3: np.matmul (also batched)
start = time.perf_counter()
result_matmul = np.matmul(X, W)
time_matmul = time.perf_counter() - start

print(f"Loop:    {time_loop*1000:.3f} ms")
print(f"einsum:  {time_einsum*1000:.3f} ms")
print(f"matmul:  {time_matmul*1000:.3f} ms")
print(f"Results match: {np.allclose(result_loop, result_einsum)}")

print("\nCommon einsum patterns:")
print("  'ij,jk->ik'     matrix multiply")
print("  'ii->i'         diagonal")
print("  'ij->j'         column sums")
print("  'ij,ij->ij'     element-wise multiply")
print("  'bik,bkj->bij'  batched matrix multiply")

## functools.lru_cache and cache — Memoization

Memoization stores the results of expensive function calls and returns the cached result when the same inputs occur again.

In [ ]:
import functools

# Naive recursive Fibonacci — exponential time O(2^n)
def fib_naive(n: int) -> int:
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


# With lru_cache — O(n) time, O(n) space
@functools.lru_cache(maxsize=128)
def fib_cached(n: int) -> int:
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)


# functools.cache (Python 3.9+) — same as lru_cache(maxsize=None)
@functools.cache
def fib_cache(n: int) -> int:
    if n <= 1:
        return n
    return fib_cache(n - 1) + fib_cache(n - 2)


print("=== Fibonacci benchmark ===")

start = time.perf_counter()
result = fib_naive(33)
time_naive = time.perf_counter() - start
print(f"fib_naive(33):   {time_naive:.4f}s  result={result}")

start = time.perf_counter()
result = fib_cached(33)
time_cached = time.perf_counter() - start
print(f"fib_cached(33):  {time_cached:.6f}s  result={result}")

print(f"Speedup: {time_naive/time_cached:.0f}x")

# Show cache info
print("\nlru_cache info:", fib_cached.cache_info())

# Cache parameters: maxsize controls LRU eviction
@functools.lru_cache(maxsize=256)   # keep last 256 unique inputs
def expensive_computation(x: int, y: int) -> float:
    # Simulate expensive work
    return sum(i ** 0.5 for i in range(x * y))

print("\nexpensive_computation(100, 200):", expensive_computation(100, 200))
print("Cache info:", expensive_computation.cache_info())
print("Second call (from cache):")
%timeit expensive_computation(100, 200)

In [ ]:
# functools.cached_property — lazy class attribute

class DataAnalyser:
    def __init__(self, data: list[float]) -> None:
        self._data = data

    @functools.cached_property
    def mean(self) -> float:
        """Computed once and cached on the instance."""
        print("  Computing mean...")
        return sum(self._data) / len(self._data)

    @functools.cached_property
    def std(self) -> float:
        """Standard deviation — also computed once."""
        print("  Computing std...")
        m = self.mean  # reuses cached mean
        variance = sum((x - m) ** 2 for x in self._data) / len(self._data)
        return variance ** 0.5


import random
data = [random.gauss(100, 15) for _ in range(100_000)]
analyser = DataAnalyser(data)

print("First access:")
print(f"  mean = {analyser.mean:.2f}")
print(f"  std  = {analyser.std:.2f}")

print("\nSecond access (from cache — no recomputation):")
print(f"  mean = {analyser.mean:.2f}")
print(f"  std  = {analyser.std:.2f}")

## Redis Caching Pattern

For web applications, `functools.lru_cache` only works within a single process. Redis is a fast in-memory data store used for distributed caching.

In [ ]:
# Install redis client (we'll show the pattern without a running server)
!pip install redis --quiet

import json
import hashlib
from typing import Any, Callable

# Redis caching decorator pattern
def redis_cache(redis_client, ttl_seconds: int = 300):
    """Decorator that caches function results in Redis."""
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs) -> Any:
            # Create a unique cache key from function name + arguments
            key_data = f"{func.__name__}:{args}:{sorted(kwargs.items())}"
            cache_key = hashlib.md5(key_data.encode()).hexdigest()

            # Try to get from cache
            try:
                cached = redis_client.get(cache_key)
                if cached:
                    print(f"  [CACHE HIT] {func.__name__}")
                    return json.loads(cached)
            except Exception:
                pass  # If Redis is down, fall through to real call

            # Cache miss — call the real function
            print(f"  [CACHE MISS] {func.__name__} — calling function")
            result = func(*args, **kwargs)

            # Store result in Redis with TTL
            try:
                redis_client.setex(cache_key, ttl_seconds, json.dumps(result))
            except Exception:
                pass

            return result
        return wrapper
    return decorator


# Usage example (needs a running Redis server — omit actual connection in Colab)
redis_usage = """
import redis

# Connect to Redis
r = redis.Redis(host='localhost', port=6379, db=0)

@redis_cache(r, ttl_seconds=60)
def get_user_from_db(user_id: int) -> dict:
    # Expensive database query
    return db.query(f"SELECT * FROM users WHERE id = {user_id}")

# First call: cache miss, queries the DB
user = get_user_from_db(42)    # [CACHE MISS] — hits DB

# Second call: cache hit, returns from Redis
user = get_user_from_db(42)    # [CACHE HIT] — <1ms
"""
print("Redis caching pattern:")
print(redis_usage)

## joblib.Memory — Disk Caching for Slow Functions

`joblib.Memory` is like `lru_cache` but stores results on disk, persisting between Python sessions. Perfect for expensive data preprocessing.

In [ ]:
from joblib import Memory
import time
import numpy as np

# Set up disk cache in a temp directory
cache_dir = "/tmp/joblib_cache"
memory = Memory(cache_dir, verbose=1)


@memory.cache
def slow_data_processing(n: int, seed: int = 42) -> np.ndarray:
    """Simulate expensive data preprocessing (e.g., feature engineering)."""
    print(f"  [RUNNING] slow_data_processing(n={n}, seed={seed})")
    rng = np.random.default_rng(seed)
    data = rng.random((n, 100))
    # Simulate expensive computation
    time.sleep(0.5)
    return (data - data.mean(axis=0)) / data.std(axis=0)  # normalise


print("=== First call (computes and caches) ===")
start = time.perf_counter()
result1 = slow_data_processing(1000)
elapsed1 = time.perf_counter() - start
print(f"  Time: {elapsed1:.3f}s  Shape: {result1.shape}")

print("\n=== Second call (loads from disk cache) ===")
start = time.perf_counter()
result2 = slow_data_processing(1000)
elapsed2 = time.perf_counter() - start
print(f"  Time: {elapsed2:.3f}s  Shape: {result2.shape}")
print(f"  Speedup: {elapsed1/max(elapsed2, 0.001):.1f}x")
print(f"  Results identical: {np.allclose(result1, result2)}")

print("\n=== Different arguments — cache miss ===")
result3 = slow_data_processing(1000, seed=99)

In [ ]:
# Summary of caching strategies
print("=" * 60)
print("CACHING STRATEGY COMPARISON")
print("=" * 60)
strategies = [
    ("functools.lru_cache",   "In-memory, single process, LRU eviction", "Pure functions with hashable args"),
    ("functools.cache",       "In-memory, single process, unbounded",    "Same as lru_cache with maxsize=None"),
    ("functools.cached_property", "Per-instance, in-memory",            "Expensive computed class attributes"),
    ("joblib.Memory",         "On-disk, persists across restarts",       "Slow data pipelines / ML preprocessing"),
    ("Redis",                 "Distributed, shared across processes",    "Web APIs, microservices, sessions"),
    ("Memcached",             "Distributed, high-speed, no persistence", "Simple key-value, high-throughput APIs"),
]
for name, storage, when in strategies:
    print(f"  {name:<28} {storage:<45} → {when}")

## Practice Exercises

**Exercise 1 — Profile and optimize**
The function below is slow. Profile it with `%lprun`, identify the bottleneck, then rewrite it using NumPy vectorization. Benchmark both versions with `%timeit` and report the speedup.
```python
def compute_distances(points: list[tuple[float, float]]) -> list[float]:
    """Return the Euclidean distance of each point from the origin."""
    distances = []
    for x, y in points:
        distances.append((x**2 + y**2) ** 0.5)
    return distances

# Test data
import random
points = [(random.uniform(-100, 100), random.uniform(-100, 100)) for _ in range(100_000)]
```

**Exercise 2 — Memoization comparison**
Implement `count_paths(m, n)` which counts the number of unique paths in an m x n grid (moving only right or down). Implement three versions:
1. Naive recursive (no caching)
2. With `@functools.lru_cache`
3. With dynamic programming (bottom-up)

Benchmark all three for `count_paths(15, 15)` and compare.

**Exercise 3 — joblib pipeline cache**
Write a data pipeline with three stages: `load_data(n)` → `transform(data)` → `aggregate(transformed)`. Cache each stage with `joblib.Memory`. Then simulate changing only the `aggregate` function and verify that the first two stages are loaded from cache while only `aggregate` is recomputed.